# 01 - Tri-mode forward 회계

**학습 목표**: AR, block diffusion, linear self-speculation이 같은 길이를 만들 때 필요한 forward와 commit 수를 분리합니다. 아래 숫자는 설명용이며 논문 benchmark 재현이 아닙니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import math

def mode_accounting(output_tokens, block_size, diffusion_tpf, accepted_prefix):
    ar_forwards = output_tokens
    diffusion_forwards = math.ceil(output_tokens / diffusion_tpf)
    ss_cycles = math.ceil(output_tokens / accepted_prefix)
    # 선형 self-speculation은 매 cycle마다 draft와 verify 두 번의 forward가 필요합니다.
    ss_forwards = 2 * ss_cycles
    return {
        'AR': (ar_forwards, output_tokens / ar_forwards),
        'diffusion': (diffusion_forwards, output_tokens / diffusion_forwards),
        'linear self-spec': (ss_forwards, output_tokens / ss_forwards),
    }

result = mode_accounting(128, block_size=32, diffusion_tpf=3.0, accepted_prefix=7.0)
for mode, (forwards, effective_tpf) in result.items():
    print(f'{mode:18s} forwards={forwards:3d} effective TPF={effective_tpf:0.2f}')
assert result['AR'][1] == 1

## 읽는 법

Self-speculation의 acceptance length가 7이어도 두 forward를 쓰므로 real TPF는 약 3.5입니다. quadratic mode와 diffusion mode는 forward당 계산량이 달라 TPF만으로 latency 순위를 정할 수 없습니다.